# USOVA3D Ovary Segmentation: 2D U-Net (Kaggle)

Trains a 2D U-Net on axial slices from the USOVA3D training volumes,
for direct comparison against the classical Random Forest pipeline
(RESULTS_LOG.md: 16-fold LOO-CV, Dice 0.785 +/- 0.073).

**Before running**: upload the `data/cnn_slices/` folder (16 .npz files
from `scripts/09_export_slices_for_cnn.py`) as a Kaggle Dataset, attach
it to this notebook, and enable GPU (T4 x2) under Settings.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import glob
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

DATA_DIR = '/data/cnn_slices'  # adjust to match your uploaded dataset path
IMG_SIZE = 256

## Load data, grouped by volume (for leakage-free CV splitting)

In [ ]:
def load_all_volumes():
    volumes = {}
    for f in sorted(glob.glob(os.path.join(DATA_DIR, '*_slices.npz'))):
        data = np.load(f)
        vol_id = int(data['vol_id'])
        volumes[vol_id] = {
            'images': data['images'],   # (n_slices, H, W), original size, varies per volume
            'masks': data['masks'],
            'orig_shape': data['images'].shape[1:],
        }
        print(f"vol{vol_id}: {len(data['images'])} slices, shape {data['images'].shape[1:]}")
    return volumes

all_volumes = load_all_volumes()
VOL_IDS = sorted(all_volumes.keys())
print(f"\nTotal volumes: {len(VOL_IDS)}")

## Dataset: resizes slices to a fixed size for batching

In [ ]:
class SliceDataset(Dataset):
    def __init__(self, volumes, vol_ids, img_size=IMG_SIZE, augment=False):
        self.samples = []
        for vid in vol_ids:
            v = volumes[vid]
            for img, mask in zip(v['images'], v['masks']):
                self.samples.append((img, mask, vid, img.shape))
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, mask, vid, orig_shape = self.samples[idx]
        img_r = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        mask_r = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        img_r = img_r.astype(np.float32) / 255.0
        if self.augment and np.random.rand() > 0.5:
            img_r = np.fliplr(img_r).copy()
            mask_r = np.fliplr(mask_r).copy()

        img_t = torch.from_numpy(img_r).unsqueeze(0)
        mask_t = torch.from_numpy(mask_r.astype(np.float32)).unsqueeze(0)
        return img_t, mask_t, vid

## U-Net (standard encoder-decoder with skip connections)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base*2)
        self.enc3 = DoubleConv(base*2, base*4)
        self.enc4 = DoubleConv(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base*8, base*16)
        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.dec4 = DoubleConv(base*16, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConv(base*2, base)
        self.out_conv = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)

## Loss (Dice + BCE, standard combo for segmentation) and training loop

In [ ]:
def dice_loss(pred_logits, target, eps=1e-6):
    pred = torch.sigmoid(pred_logits)
    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()

def combined_loss(pred_logits, target):
    bce = F.binary_cross_entropy_with_logits(pred_logits, target)
    dl = dice_loss(pred_logits, target)
    return bce + dl

def train_model(train_loader, n_epochs=30, lr=1e-3):
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss = combined_loss(preds, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch+1) % 5 == 0:
            print(f"  epoch {epoch+1}/{n_epochs}: loss={total_loss/len(train_loader):.4f}")
    return model

## 4-fold grouped cross-validation

Splits the 16 volumes into 4 groups (grouped, not random slice-level
split, to avoid leakage between slices of the same volume). Reconstructs
each held-out volume's predicted mask at ORIGINAL resolution to compute
a fair, directly comparable Dice/IoU against the classical baseline.

In [ ]:
def volumetric_dice_iou(pred_mask, gt_mask):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    intersection = (pred & gt).sum()
    union = (pred | gt).sum()
    dice = 2*intersection / (pred.sum() + gt.sum() + 1e-8)
    iou = intersection / (union + 1e-8)
    return dice, iou

def evaluate_volume(model, volume_data, img_size=IMG_SIZE):
    model.eval()
    images = volume_data['images']
    orig_h, orig_w = volume_data['orig_shape']
    pred_slices = []
    with torch.no_grad():
        for img in images:
            img_r = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
            img_t = torch.from_numpy(img_r.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(device)
            pred_logits = model(img_t)
            pred_prob = torch.sigmoid(pred_logits).cpu().numpy()[0, 0]
            pred_mask = (pred_prob > 0.5).astype(np.uint8)
            pred_mask_orig = cv2.resize(pred_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            pred_slices.append(pred_mask_orig)
    return np.stack(pred_slices)

np.random.seed(42)
shuffled = np.random.permutation(VOL_IDS)
groups = np.array_split(shuffled, 4)

import json
import os

results_path = '/kaggle/working/unet_cv_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        all_results = json.load(f)
    done_vol_ids = {r['vol_id'] for r in all_results}
    print(f"Resuming: {len(done_vol_ids)} volumes already evaluated: {done_vol_ids}")
else:
    all_results = []
    done_vol_ids = set()

for fold_idx, test_group in enumerate(groups):
    test_ids = list(test_group)
    if all(v in done_vol_ids for v in test_ids):
        print(f"Fold {fold_idx+1}/4 already complete, skipping")
        continue
    train_ids = [v for v in VOL_IDS if v not in test_ids]
    print(f"\n=== Fold {fold_idx+1}/4: test volumes {test_ids} ===")

    train_ds = SliceDataset(all_volumes, train_ids, augment=True)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)

    model = train_model(train_loader, n_epochs=30)

    for test_vid in test_ids:
        pred_volume = evaluate_volume(model, all_volumes[test_vid])
        gt_volume = all_volumes[test_vid]['masks']
        dice, iou = volumetric_dice_iou(pred_volume, gt_volume)
        all_results.append({'vol_id': int(test_vid), 'dice': float(dice), 'iou': float(iou)})
        print(f"  vol{test_vid}: Dice={dice:.3f}, IoU={iou:.3f}")

        with open(results_path, 'w') as f:
            json.dump(all_results, f, indent=2)

    torch.save(model.state_dict(), f'/kaggle/working/unet_fold{fold_idx}.pt')
    print(f"Fold {fold_idx+1}/4 complete, progress saved.")

## Final comparison against the classical Random Forest baseline

In [ ]:
import pandas as pd

results_df = pd.DataFrame(all_results)
results_df.to_csv('/kaggle/working/unet_cv_results.csv', index=False)

print(f"U-Net (2D, 4-fold grouped CV): Dice = {results_df['dice'].mean():.3f} +/- {results_df['dice'].std():.3f}")
print(f"U-Net (2D, 4-fold grouped CV): IoU  = {results_df['iou'].mean():.3f} +/- {results_df['iou'].std():.3f}")
print()
print(f"Classical Random Forest (16-fold LOO-CV): Dice = 0.785 +/- 0.073")
print(f"(Note: 4-fold vs 16-fold CV granularity differs, and 2D-slice-based")
print(f"prediction vs. genuinely 3D features is a real methodological")
print(f"difference worth noting explicitly when reporting this comparison)")

results_df